In [16]:
!pip install numpy tensorflow

In [17]:
import json
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Embedding,
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint
)

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [19]:
CAPTIONS_PATH = "/Users/ameer/Deep_Learning/generated_captions.json"

with open(CAPTIONS_PATH, "r", encoding="utf-8") as f:
    captions_data = json.load(f)

print("Total Captions:", len(captions_data))

Total Captions: 8091


In [20]:
def create_question(caption):

    caption = caption.lower()

    if "running" in caption:
        return "what is running"

    elif "climbing" in caption:
        return "what is the person climbing"

    elif "swimming" in caption:
        return "who is swimming"

    elif "playing" in caption:
        return "what is the person playing"

    elif "standing" in caption:
        return "who is standing"

    else:
        return "what is happening"

In [21]:
captions = []
questions = []

for img, caption in captions_data.items():

    question = create_question(caption)

    captions.append(caption)

    questions.append(
        "startseq " + question + " endseq"
    )

print("Dataset Size:", len(captions))

for i in range(5):

    print("\nCaption:", captions[i])
    print("Question:", questions[i])

Dataset Size: 8091

Caption: a woman in a red shirt is standing in front of a building
Question: startseq who is standing endseq

Caption: a dog is running through a field
Question: startseq what is running endseq

Caption: a young girl in a red shirt is standing on a bench with a man in a red shirt
Question: startseq who is standing endseq

Caption: a man in a red shirt is standing in front of a man in a red shirt
Question: startseq who is standing endseq

Caption: a man in a red shirt is standing in front of a building
Question: startseq who is standing endseq


In [22]:
tokenizer = Tokenizer(
    num_words=5000,
    oov_token="<unk>"
)

tokenizer.fit_on_texts(
    captions + questions
)

word_to_idx = tokenizer.word_index
idx_to_word = tokenizer.index_word

vocab_size = len(word_to_idx) + 1

print("Vocabulary Size:", vocab_size)

Vocabulary Size: 133


In [23]:
caption_sequences = tokenizer.texts_to_sequences(captions)

question_sequences = tokenizer.texts_to_sequences(questions)

max_caption_len = max(
    len(seq) for seq in caption_sequences
)

max_question_len = max(
    len(seq) for seq in question_sequences
)

print("Max Caption Length:", max_caption_len)
print("Max Question Length:", max_question_len)

Max Caption Length: 35
Max Question Length: 7


In [24]:
encoder_input_data = []
decoder_input_data = []
decoder_output_data = []

for caption_seq, question_seq in zip(
    caption_sequences,
    question_sequences
):

    for i in range(1, len(question_seq)):

        in_seq = question_seq[:i]

        out_seq = question_seq[i]

        in_seq = pad_sequences(
            [in_seq],
            maxlen=max_question_len,
            padding='post'
        )[0]

        caption_seq_padded = pad_sequences(
            [caption_seq],
            maxlen=max_caption_len,
            padding='post'
        )[0]

        out_seq = to_categorical(
            out_seq,
            num_classes=vocab_size
        )

        encoder_input_data.append(
            caption_seq_padded
        )

        decoder_input_data.append(
            in_seq
        )

        decoder_output_data.append(
            out_seq
        )

encoder_input_data = np.array(
    encoder_input_data
)

decoder_input_data = np.array(
    decoder_input_data
)

decoder_output_data = np.array(
    decoder_output_data
)

print("Encoder Shape:", encoder_input_data.shape)
print("Decoder Shape:", decoder_input_data.shape)

Encoder Shape: (35250, 35)
Decoder Shape: (35250, 7)


In [25]:
EMBEDDING_DIM = 256
LSTM_UNITS = 256

# Encoder
encoder_inputs = Input(
    shape=(max_caption_len,)
)

encoder_embedding = Embedding(
    vocab_size,
    EMBEDDING_DIM,
    mask_zero=True
)(encoder_inputs)

encoder_lstm = LSTM(
    LSTM_UNITS,
    return_state=True
)

_, state_h, state_c = encoder_lstm(
    encoder_embedding
)

encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(
    shape=(max_question_len,)
)

decoder_embedding = Embedding(
    vocab_size,
    EMBEDDING_DIM,
    mask_zero=True
)(decoder_inputs)

decoder_lstm = LSTM(
    LSTM_UNITS
)

decoder_outputs = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

decoder_outputs = Dropout(0.5)(
    decoder_outputs
)

decoder_outputs = Dense(
    vocab_size,
    activation='softmax'
)(decoder_outputs)

# Final Model
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 35)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 35, 256)   │     34,048 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 35)        │          0 │ input_layer_2[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 7, 256)    │     34,048 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, 256),     │    525,312 │ embedding_2[0][0… │
│                     │ (None, 256),      │            │ not_equal_2[0][0] │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ (None, 256)       │    525,312 │ embedding_3[0][0… │
│                     │                   │            │ lstm_2[0][1],     │
│                     │                   │            │ lstm_2[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ lstm_3[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 133)       │     34,181 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,152,901 (4.40 MB)

 Trainable params: 1,152,901 (4.40 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
callbacks = [

    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    ),

    ModelCheckpoint(
        'question_generator.keras',
        monitor='val_loss',
        save_best_only=True
    )
]

history = model.fit(

    [encoder_input_data, decoder_input_data],

    decoder_output_data,

    epochs=15,

    batch_size=64,

    callbacks=callbacks
)

Epoch 1/15
551/551 ━━━━━━━━━━━━━━━━━━━━ 26s 47ms/step - accuracy: 0.9998 - loss: 6.6161e-04
Epoch 2/15
551/551 ━━━━━━━━━━━━━━━━━━━━ 26s 48ms/step - accuracy: 1.0000 - loss: 1.4428e-04
Epoch 3/15
551/551 ━━━━━━━━━━━━━━━━━━━━ 28s 51ms/step - accuracy: 1.0000 - loss: 1.5378e-04
Epoch 4/15
551/551 ━━━━━━━━━━━━━━━━━━━━ 28s 51ms/step - accuracy: 1.0000 - loss: 4.9318e-05
Epoch 5/15
551/551 ━━━━━━━━━━━━━━━━━━━━ 29s 53ms/step - accuracy: 1.0000 - loss: 3.5856e-05
Epoch 6/15
551/551 ━━━━━━━━━━━━━━━━━━━━ 31s 57ms/step - accuracy: 1.0000 - loss: 2.1820e-05
Epoch 7/15
551/551 ━━━━━━━━━━━━━━━━━━━━ 34s 62ms/step - accuracy: 1.0000 - loss: 1.4892e-05
Epoch 8/15
551/551 ━━━━━━━━━━━━━━━━━━━━ 34s 61ms/step - accuracy: 1.0000 - loss: 1.2171e-05
Epoch 9/15
551/551 ━━━━━━━━━━━━━━━━━━━━ 33s 60ms/step - accuracy: 1.0000 - loss: 9.0147e-06
Epoch 10/15
551/551 ━━━━━━━━━━━━━━━━━━━━ 33s 60ms/step - accuracy: 1.0000 - loss: 6.7069e-06
Epoch 11/15
551/551 ━━━━━━━━━━━━━━━━━━━━ 34s 61ms/step - accuracy: 1.0000 - los

In [28]:
tokenizer_json = tokenizer.to_json()

with open(
    'question_tokenizer.json',
    'w',
    encoding='utf-8'
) as f:

    f.write(tokenizer_json)

print("Tokenizer Saved.")

Tokenizer Saved.


In [29]:
def generate_question(caption):

    caption_seq = tokenizer.texts_to_sequences(
        [caption]
    )[0]

    caption_seq = pad_sequences(
        [caption_seq],
        maxlen=max_caption_len,
        padding='post'
    )

    question = 'startseq'

    for _ in range(max_question_len):

        question_seq = tokenizer.texts_to_sequences(
            [question]
        )[0]

        question_seq = pad_sequences(
            [question_seq],
            maxlen=max_question_len,
            padding='post'
        )

        pred = model.predict(
            [caption_seq, question_seq],
            verbose=0
        )

        pred_index = np.argmax(pred)

        next_word = idx_to_word.get(
            pred_index,
            ''
        )

        if next_word == 'endseq':
            break

        if next_word == '':
            break

        question += ' ' + next_word

    return question.replace(
        'startseq',
        ''
    ).strip()

In [30]:
test_caption = "a dog is running through a field"

generated_question = generate_question(
    test_caption
)

print("Caption:")
print(test_caption)

print("\nGenerated Question:")
print(generated_question)

Caption:
a dog is running through a field

Generated Question:
what is running
